In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

### Carga

In [16]:
#features que descarto
# features_to_drop = ['WFO', 'SOURCE']
features_to_drop = ['WFO', 'SOURCE','CZ_TYPE',
                    'TIME_OF_DAY','MAGNITUDE_TYPE','YEAR',
                    'DAY_OF_WEEK','DAY_OF_YEAR','END_LAT',
                    'END_LON','BEGIN_LAT','BEGIN_LON',
                    'HAS_TRACK_DISTANCE','HAS_TORNADO_DATA','HAS_MAGNITUDE']

In [17]:
# ---------------------------------------------------------
# 1. Load the Datasets
# ---------------------------------------------------------
print("Loading datasets...")

DATA_PATH = '../tp/processed_datasets/'
X_TRAIN_PATH = DATA_PATH + 'X_train_catboost.parquet'
X_VAL_PATH = DATA_PATH + 'X_val_catboost.parquet'
X_TEST_PATH = DATA_PATH + 'X_test_catboost.parquet'

Y_TRAIN_PATH = DATA_PATH + 'y_train.csv'
Y_VAL_PATH = DATA_PATH + 'y_val.csv'
Y_TEST_PATH = DATA_PATH + 'y_test.csv'
# Using fastparquet to avoid any ArrowKeyError version conflicts

# Load the data and drop the columns in one go
X_train = pd.read_parquet(X_TRAIN_PATH, engine="fastparquet").drop(columns=features_to_drop)
X_val = pd.read_parquet(X_VAL_PATH, engine="fastparquet").drop(columns=features_to_drop)
X_test = pd.read_parquet(X_TEST_PATH, engine="fastparquet").drop(columns=features_to_drop)

y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()
y_val = pd.read_csv(Y_VAL_PATH).squeeze()
y_test = pd.read_csv(Y_TEST_PATH).squeeze()

print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Validation shapes: X={X_val.shape}, y={y_val.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}\n")

categorical_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
#Added day of week since its encoded but not ordinal
# categorical_features.extend(['DAY_OF_WEEK'])
print(f"Detected {len(categorical_features)} categorical features: {categorical_features}")

Loading datasets...
Train shapes: X=(876565, 17), y=(876565,)
Validation shapes: X=(156915, 17), y=(156915,)
Test shapes: X=(115998, 17), y=(115998,)

Detected 5 categorical features: ['EVENT_TYPE', 'STATE', 'REGION', 'SEASON', 'FLOOD_CAUSE']


In [18]:
TORNADO_EVENTS = ['Tornado']

WIND_EVENTS = [
    'Thunderstorm Wind', 'Cold/Wind Chill', 'Extreme Cold/Wind Chill', 
    'High Wind', 'Marine High Wind', 'Marine Strong Wind', 
    'Marine Thunderstorm Wind', 'Strong Wind'
]

HAIL_EVENTS = ['Hail', 'Marine Hail']

FLOOD_EVENTS = ['Coastal Flood', 'Flash Flood', 'Flood', 'Lakeshore Flood']

TOP_EVENTS = [
    'Hurricane (Typhoon)', 'Storm Surge/Tide', 'Wildfire', 'Tropical Storm', 
    'Tsunami', 'Ice Storm', 'Drought', 'Debris Flow', 'Frost/Freeze', 
    'Blizzard', 'Tropical Depression', 'Volcanic Ash'
]

# Combine all lists to easily identify the "Rest" category
ALL_KNOWN_EVENTS = set(TORNADO_EVENTS + WIND_EVENTS + HAIL_EVENTS + FLOOD_EVENTS + TOP_EVENTS)

def get_event_subset(X, y, event_list=None, is_rest=False):
    """
    Filters the X and y dataframes based on EVENT_TYPE.
    Returns explicit copies to prevent SettingWithCopyWarnings later.
    """
    if is_rest:
        # Keep rows where EVENT_TYPE is NOT IN the known events
        mask = ~X['EVENT_TYPE'].isin(ALL_KNOWN_EVENTS)
    else:
        # Keep rows where EVENT_TYPE matches the provided list
        mask = X['EVENT_TYPE'].isin(event_list)
        
    return X[mask].copy(), y[mask].copy()

# ---------------------------------------------------------
# 3. Generate the Splits (Using Training Data as the example)
# ---------------------------------------------------------
print("Splitting datasets by Event Type...")

# Tornado
X_tornado_train, y_tornado_train = get_event_subset(X_train, y_train, TORNADO_EVENTS)
X_tornado_val, y_tornado_val = get_event_subset(X_val, y_val, TORNADO_EVENTS)
# Wind
X_wind_train, y_wind_train = get_event_subset(X_train, y_train, WIND_EVENTS)
X_wind_val, y_wind_val = get_event_subset(X_val, y_val, WIND_EVENTS)
# Hail
X_hail_train, y_hail_train = get_event_subset(X_train, y_train, HAIL_EVENTS)
X_hail_val, y_hail_val = get_event_subset(X_val, y_val, HAIL_EVENTS)
# Flood
X_flood_train, y_flood_train = get_event_subset(X_train, y_train, FLOOD_EVENTS)
X_flood_val, y_flood_val = get_event_subset(X_val, y_val, FLOOD_EVENTS)
# Top
X_top_train, y_top_train = get_event_subset(X_train, y_train, TOP_EVENTS)
X_top_val, y_top_val = get_event_subset(X_val, y_val, TOP_EVENTS)
# Rest
X_rest_train, y_rest_train = get_event_subset(X_train, y_train, is_rest=True)
X_rest_val, y_rest_val = get_event_subset(X_val, y_val, is_rest=True)
# ---------------------------------------------------------
# Optional: Verify the shapes to ensure no data was dropped
# ---------------------------------------------------------
total_original = len(X_train)
total_split = len(X_tornado_train) + len(X_wind_train) + len(X_hail_train) + len(X_flood_train) + len(X_top_train) + len(X_rest_train)

print(f"Original Train Rows: {total_original}")
# Note: total_split might be slightly higher than total_original if 'Lakeshore Flood' is kept in two lists!
print(f"Total Rows in Splits: {total_split}") 


# ['Tornado'] -> Tornado
# ['Thunderstorm Wind', 'Cold/Wind Chill','Extreme Cold/Wind Chill','High Wind','Marine High Wind','Marine Strong Wind','Marine Thunderstorm Wind','Strong Wind','Thunderstorm Wind'] -> Wind
# ['Hail','Marine Hail'] -> Hail
# ['Coastal Flood','Flash Flood','Flood','Lakeshore Flood'] -> Flood
# [Hurricane (Typhoon)   
# Storm Surge/Tide       
# Wildfire               
# Tropical Storm         
# Tsunami                
# Ice Storm              
# Drought                
# Debris Flow            
# Frost/Freeze           
# Lakeshore Flood        
# Blizzard               
# Tropical Depression    
# Volcanic Ash] -> Top

# if they dont belong to any of the listed then -> rest

Splitting datasets by Event Type...
Original Train Rows: 876565
Total Rows in Splits: 876565


### Entrenamiento

In [19]:
train_params = {
    "random_seed": 42,
    "verbose": False,
    "task_type": "GPU",
    "eval_metric": "TotalF1:average=Macro"
}

# ---------------------------------------------------------
# TORNADO MODEL
# ---------------------------------------------------------
# MOVE ignored_features UP HERE:
tornado_model = CatBoostClassifier(**train_params, ignored_features=['MAGNITUDE','FLOOD_CAUSE'])

tornado_model.fit(
    X_tornado_train,  
    y_tornado_train,  
    cat_features=categorical_features,
    eval_set=[(X_tornado_val, y_tornado_val)], 
    early_stopping_rounds=100,
    use_best_model=True
)

# ---------------------------------------------------------
# WIND MODEL
# ---------------------------------------------------------
wind_model = CatBoostClassifier(**train_params, ignored_features=['TOR_SCALE_NUM','TOR_LENGTH','TOR_WIDTH','TOR_AREA_KM2','FLOOD_CAUSE'])

wind_model.fit(
    X_wind_train,     
    y_wind_train,     
    cat_features=categorical_features,
    eval_set=[(X_wind_val, y_wind_val)],       
    early_stopping_rounds=100,
    use_best_model=True
)

# ---------------------------------------------------------
# HAIL MODEL
# ---------------------------------------------------------
hail_model = CatBoostClassifier(**train_params, ignored_features=['TOR_SCALE_NUM','TOR_LENGTH','TOR_WIDTH','TOR_AREA_KM2','FLOOD_CAUSE'])

hail_model.fit(
    X_hail_train,     
    y_hail_train,     
    cat_features=categorical_features,
    eval_set=[(X_hail_val, y_hail_val)],       
    early_stopping_rounds=100,
    use_best_model=True
)

# ---------------------------------------------------------
# FLOOD MODEL
# ---------------------------------------------------------
flood_model = CatBoostClassifier(**train_params, ignored_features=['TOR_SCALE_NUM','TOR_LENGTH','TOR_WIDTH','TOR_AREA_KM2','MAGNITUDE'])

flood_model.fit(
    X_flood_train,    
    y_flood_train,    
    cat_features=categorical_features,
    eval_set=[(X_flood_val, y_flood_val)],     
    early_stopping_rounds=100,
    use_best_model=True
)

# ---------------------------------------------------------
# TOP MODEL
# ---------------------------------------------------------
top_model = CatBoostClassifier(**train_params, ignored_features=['TOR_SCALE_NUM','TOR_LENGTH','TOR_WIDTH','TOR_AREA_KM2','FLOOD_CAUSE','MAGNITUDE'])

top_model.fit(
    X_top_train,      
    y_top_train,      
    cat_features=categorical_features,
    eval_set=[(X_top_val, y_top_val)],         
    early_stopping_rounds=100,
    use_best_model=True
)

# ---------------------------------------------------------
# REST MODEL
# ---------------------------------------------------------
rest_model = CatBoostClassifier(**train_params, ignored_features=['TOR_SCALE_NUM','TOR_LENGTH','TOR_WIDTH','TOR_AREA_KM2','FLOOD_CAUSE'])

rest_model.fit(
    X_rest_train,     
    y_rest_train,     
    cat_features=categorical_features,
    eval_set=[(X_rest_val, y_rest_val)],       
    early_stopping_rounds=100,
    use_best_model=True
)

CatBoostClassifier(eval_metric='TotalF1:average=Macro', ignored_features=['TOR_SCALE_NUM', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_AREA_KM2', 'FLOOD_CAUSE'], random_seed=42, task_type='GPU', verbose=False)

### Prediccion en Test

In [20]:
def composed_predict(X, models_dict):
    """
    Routes rows to specific models based on EVENT_TYPE and returns a combined prediction Series.
    """
    # Create an empty Series to hold our predictions, perfectly matching X's index
    final_predictions = pd.Series(index=X.index, dtype=object)
    
    # Define our routing logic mapping
    routes = [
        (TORNADO_EVENTS, models_dict['tornado']),
        (WIND_EVENTS, models_dict['wind']),
        (HAIL_EVENTS, models_dict['hail']),
        (FLOOD_EVENTS, models_dict['flood']),
        (TOP_EVENTS, models_dict['top'])
    ]
    
    # 1. Route the known events
    for event_list, model in routes:
        # Find rows that belong to this category
        mask = X['EVENT_TYPE'].isin(event_list)
        
        # If we have rows matching this category, predict and store them
        if mask.any():
            preds = model.predict(X[mask])
            # Flatten because CatBoost sometimes returns 2D arrays (e.g., [[0], [1]])
            final_predictions.loc[mask] = preds.flatten()
            
    # 2. Route the "Rest" category
    mask_rest = ~X['EVENT_TYPE'].isin(ALL_KNOWN_EVENTS)
    if mask_rest.any():
        preds = models_dict['rest'].predict(X[mask_rest])
        final_predictions.loc[mask_rest] = preds.flatten()
        
    # Ensure no rows were missed (sanity check)
    if final_predictions.isna().any():
        print("WARNING: Some rows did not receive a prediction!")
        
    return final_predictions.astype(int) # Cast back to your integer class labels (0, 1, 2, 3, 4)

In [21]:

# Pack the models into a dictionary for clean passing
my_models = {
    'tornado': tornado_model,
    'wind': wind_model,
    'hail': hail_model,
    'flood': flood_model,
    'top': top_model,
    'rest': rest_model
}

print("Running composed model predictions on full test set...")

# Get the combined predictions
y_pred_composed = composed_predict(X_test, my_models)

# Print overall performance metrics
print("=========================================")
print(" OVERALL COMPOSED MODEL PERFORMANCE")
print(f" (Total Test Samples: {len(y_test)})")
print("=========================================")

overall_accuracy = accuracy_score(y_test, y_pred_composed)
print(f"Final Overall Test Accuracy: {overall_accuracy:.4f}\n")

print("Overall Classification Report:")
print(classification_report(y_test, y_pred_composed, zero_division=0))

Running composed model predictions on full test set...
 OVERALL COMPOSED MODEL PERFORMANCE
 (Total Test Samples: 115998)
Final Overall Test Accuracy: 0.7962

Overall Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     86950
           1       0.65      0.61      0.63     17859
           2       0.26      0.29      0.27      7736
           3       0.24      0.21      0.22      2274
           4       0.34      0.19      0.24      1179

    accuracy                           0.80    115998
   macro avg       0.48      0.44      0.45    115998
weighted avg       0.79      0.80      0.79    115998

